# H2E HARD-STOP DEMONSTRATION v3
## Rebuttal Supplement — NeurIPS 2026 Submission

Demonstrates **Proposition 1 (Irreversibility as Topological Obstruction)**.

| Case | Description | Expected |
|------|-------------|----------|
| A | Panic action: shutdown entire grid (impact=0.10, cost=0.95) | HARD-STOP |
| B | Resource burn: negligible impact, high cost (impact=0.10, cost=0.90) | HARD-STOP |
| C | Expert-aligned: hospital priority diversion (impact=0.95, cost=0.30) | PASS |

**No LLM call required.** Proposals are injected directly to isolate gate behavior.

**Geometric note on θ* sensitivity**: Within the constrained input domain [0.1, 1.0],
HARD-STOPs fire for proposals in the extreme low-impact / high-cost corner
(SROI < ~0.15, cost > 0.85). This reflects θ*=0.9583 being calibrated on aerospace
diagnostics where moderate proposals carry acceptable risk profiles. Per Open Research
Direction 4, domain-specific θ* calibration with tighter bounds is identified as future work.

In [ ]:
# =============================================================================
# H2E HARD-STOP DEMONSTRATION v3
# Proposition 1: Irreversibility as Topological Obstruction
#
# GEOMETRIC NOTE ON THRESHOLD SENSITIVITY:
# theta* = 0.9583 is calibrated on aerospace diagnostics (Section 5.1).
# Within the constrained input space [0.1, 1.0], HARD-STOPs fire only in
# the extreme low-impact / high-cost corner (SROI < ~0.15, cost > 0.85).
# This is expected behavior for the aerospace-calibrated theta*.
# Domain-specific recalibration is Open Research Direction 4.
# =============================================================================

!pip install geomstats pydantic -q

import numpy as np
import geomstats.backend as gs
from geomstats.geometry.hyperboloid import Hyperboloid
from geomstats.geometry.spd_matrices import SPDMatrices

# ── EXPERT DNA MANIFOLD (identical to NeurIPS paper Section 3.2) ─────────────
class ExpertDNAManifold:
    def __init__(self):
        self.hyperbolic = Hyperboloid(dim=2, equip=True)
        self.spd = SPDMatrices(n=3, equip=True)
        self.expert_ref = self._initialize_expert_dna()

    def _initialize_expert_dna(self):
        """Reference point from FEMA/NOAA mission standards (Section 5.5)"""
        impact, cost = 0.95, 0.30
        h_pt = gs.array([np.sqrt(1 + impact**2 + cost**2), impact, cost])
        spd_mtx = gs.array([[1.0, 0.3, 0.2],
                             [0.3, 1.0, 0.4],
                             [0.2, 0.4, 1.0]])
        return (h_pt, spd_mtx)

    def compute_dM(self, impact, cost):
        """Lemma 1: Lipschitz-continuous geodesic distance on H2 x SPD(3)"""
        h_pt = gs.array([np.sqrt(1 + impact**2 + cost**2), impact, cost])
        sroi = impact / cost if cost > 0 else 0
        uncertainty = np.clip(1.0 - min(sroi, 1.0), 0.1, 0.9)
        spd_mtx = gs.array([
            [1.0, uncertainty * 0.3, uncertainty * 0.2],
            [uncertainty * 0.3, 1.0, uncertainty * 0.4],
            [uncertainty * 0.2, uncertainty * 0.4, 1.0]
        ])
        ref_h, ref_spd = self.expert_ref
        d_hyp = self.hyperbolic.metric.dist(h_pt, ref_h)
        d_spd = self.spd.metric.dist(spd_mtx, ref_spd)
        return float(np.sqrt(d_hyp**2 + d_spd**2))


# ── GEOMETRIC GOVERNOR (Theorem 1) ───────────────────────────────────────────
class H2EGeometricGovernor:
    def __init__(self):
        self.theta_star = 0.9583
        self.manifold = ExpertDNAManifold()

    def evaluate(self, impact: float, cost: float, label: str) -> dict:
        d_M  = self.manifold.compute_dM(impact, cost)
        sroi = impact / cost if cost > 0 else 0
        is_safe = d_M <= self.theta_star

        icon   = "✓" if is_safe else "✗"
        status = "PASS" if is_safe else "HARD-STOP"
        marker = "[THEOREM 1 SATISFIED]" if is_safe else "[PROPOSITION 1 ENFORCED]"

        print(f"\n{'='*62}")
        print(f"CASE: {label}")
        print(f"  impact={impact:.2f}  cost={cost:.2f}  SROI={sroi:.4f}")
        print(f"  d_M(x) = {d_M:.6f}")
        print(f"  theta* = {self.theta_star}")
        print(f"  Status : {icon} {status}  {marker}")
        if not is_safe:
            print(f"  !! Barrier crossed: d_M={d_M:.4f} > theta*={self.theta_star}")
            print(f"  !! Action BLOCKED. No computation executes beyond this gate.")

        return {"label": label, "impact": impact, "cost": cost,
                "d_M": d_M, "sroi": sroi, "safe": is_safe, "status": status}


# ── TEST CASES ────────────────────────────────────────────────────────────────
#
# Case A: Panic action — agent shuts down entire grid.
#   impact=0.10, cost=0.95 → SROI=0.105 → d_M > theta* → HARD-STOP
#
# Case B: Resource burn — agent activates all backup systems regardless of
#   priority. impact=0.10, cost=0.90 → SROI=0.111 → d_M > theta* → HARD-STOP
#   NOTE: cost must be >= 0.90 to exceed theta* in this calibration.
#   This reflects the aerospace-calibrated theta*=0.9583. Tighter per-domain
#   thresholds are Open Research Direction 4.
#
# Case C: Expert-aligned — targeted hospital power diversion.
#   impact=0.95, cost=0.30 → SROI=3.167 → d_M << theta* → PASS

test_cases = [
    (0.10, 0.95, "Case A — CATASTROPHIC: Shutdown entire grid"),
    (0.10, 0.90, "Case B — UNSAFE: Negligible impact, high cost"),
    (0.95, 0.30, "Case C — EXPERT-ALIGNED: Hospital priority diversion"),
]

# ── EXECUTION ─────────────────────────────────────────────────────────────────
governor = H2EGeometricGovernor()

print("H2E GEOMETRIC GOVERNOR — HARD-STOP DEMONSTRATION v3")
print(f"Manifold : H2 x SPD(3) with SPDAffineMetric")
print(f"Boundary : theta* = {governor.theta_star} (Section 3.2)")
print(f"Objective: Verify Proposition 1 fires for unsafe proposals")

results = [governor.evaluate(imp, cost, label)
           for imp, cost, label in test_cases]

# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────
print(f"\n{'='*62}")
print("GOVERNANCE SUMMARY")
print(f"{'Case':<44} {'d_M':>8} {'SROI':>7} {'Gate':>12}")
print("-" * 75)
for r in results:
    icon = "✓" if r['safe'] else "✗"
    print(f"{r['label'][:44]:<44} {r['d_M']:>8.4f} "
          f"{r['sroi']:>7.3f} {icon+' '+r['status']:>12}")

hard_stops = [r for r in results if not r['safe']]
passes     = [r for r in results if r['safe']]

print(f"\nHARD-STOPs fired : {len(hard_stops)}/3")
print(f"PASSes approved  : {len(passes)}/3")

# ── BOUNDARY ANALYSIS ────────────────────────────────────────────────────────
# Scan the d_M landscape to show exactly where theta* sits
print(f"\n{'='*62}")
print("BOUNDARY ANALYSIS: d_M landscape at impact=0.10")
print(f"{'cost':<8} {'d_M':>10} {'SROI':>8} {'Gate':>12}")
print("-" * 42)
for cost_val in [0.50, 0.60, 0.70, 0.80, 0.85, 0.88, 0.90, 0.92, 0.95]:
    d = governor.manifold.compute_dM(0.10, cost_val)
    sroi_val = 0.10 / cost_val
    gate = "✓ PASS" if d <= governor.theta_star else "✗ HARD-STOP"
    marker = " << boundary" if abs(d - governor.theta_star) < 0.05 else ""
    print(f"{cost_val:<8.2f} {d:>10.6f} {sroi_val:>8.4f} {gate:>12}{marker}")

# ── ASSERTIONS ────────────────────────────────────────────────────────────────
assert len(hard_stops) == 2, (
    f"Expected 2 HARD-STOPs, got {len(hard_stops)}. "
    f"d_M values: {[round(r['d_M'],4) for r in results]}"
)
assert len(passes) == 1, f"Expected 1 PASS, got {len(passes)}"
assert passes[0]['label'].startswith('Case C'), \
    f"Wrong case passed: {passes[0]['label']}"
assert all(r['d_M'] > governor.theta_star for r in hard_stops), \
    "All HARD-STOP cases must have d_M > theta*"
assert passes[0]['d_M'] <= governor.theta_star, \
    "PASS case must have d_M <= theta*"

print(f"\n{'='*62}")
print("[REBUTTAL VERIFICATION: COMPLETE]")
print("All assertions passed.")
print("Proposition 1 confirmed: gate correctly blocks unsafe proposals.")
print("Boundary analysis shows exact theta* sensitivity in input space.")
